# TruLens RAG Triad: Complete Guide to Evaluating RAG Systems

**Created for**: Manager Review  
**Purpose**: Understanding and implementing TruLens RAG Triad metrics  
**Environment**: Local Ollama (phi3:mini + llama3.2)

---

## Table of Contents
1. [Introduction to RAG Triad](#section-1)
2. [Metric 1: Context Relevance](#section-2)
3. [Metric 2: Groundedness](#section-3)
4. [Metric 3: Answer Relevance](#section-4)
5. [Complete RAG Evaluation](#section-5)
6. [Comparison with RAGAS](#section-6)
7. [Best Practices](#section-7)
8. [Conclusion](#section-8)

---
# Section 1: Introduction to RAG Triad {#section-1}

## What is the RAG Triad?

The **RAG Triad** is TruLens's evaluation framework for assessing the quality and reliability of Retrieval-Augmented Generation (RAG) systems. It consists of three key metrics that evaluate different aspects of the RAG pipeline:

```
Query → [Context Relevance] → Retrieval → [Groundedness] → Generation → [Answer Relevance] → Response
```

### The Three Pillars

1. **Context Relevance** (Retrieval Quality)
   - Evaluates: Is the retrieved context relevant to the query?
   - Prevents: Irrelevant information being used for generation
   
2. **Groundedness** (Faithfulness)
   - Evaluates: Is the answer supported by the retrieved context?
   - Prevents: Hallucinations and unsupported claims
   
3. **Answer Relevance** (Response Quality)
   - Evaluates: Does the answer actually address the question?
   - Prevents: Off-topic or incomplete responses

### Why These 3 Metrics?

RAG systems can fail in multiple ways:
- ❌ **Poor Retrieval**: Returns irrelevant documents → Low Context Relevance
- ❌ **Hallucination**: Makes up facts not in context → Low Groundedness
- ❌ **Off-Topic Answer**: Doesn't answer the question → Low Answer Relevance

By achieving high scores on **all three metrics**, we can confidently assert that our RAG system is producing reliable, accurate, and relevant responses.

### How TruLens Differs from RAGAS

| Aspect | TruLens | RAGAS |
|--------|---------|-------|
| **Approach** | LLM-based evaluation | Combination of LLM + heuristics |
| **Metrics** | 3 core (RAG Triad) | 4+ metrics |
| **Transparency** | Provides reasoning chains | Scores only |
| **Flexibility** | Framework-agnostic | More opinionated |
| **Dashboard** | Built-in Streamlit UI | Separate visualization needed |

**Key Difference**: TruLens focuses on the "triad" that covers the three main failure modes of RAG, while RAGAS offers more granular metrics.

In [2]:
# %%
# Import required libraries
import sys
import requests  # ✅ Add this if not already there
from typing import List, Dict, Any
from litellm import completion  # Remove 'embedding' from here
from trulens.core import Feedback, TruSession
from trulens.apps.custom import instrument
from trulens.providers.openai import OpenAI as OpenAIProvider
import chromadb
from chromadb.config import Settings
import numpy as np

print("✅ Libraries imported successfully")


d:\eval truelense\venv\Lib\site-packages\munch\__init__.py:24: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
d:\eval truelense\venv\Lib\site-packages\trulens\core\utils\imports.py:591: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  return self.imp(name, globals, locals, fromlist, level)
C:\Users\ssb18\AppData\Local\Temp\ipykernel_16572\680821132.py:8: DeprecationWarning: from trulens.apps.custom import instrument
        is being deprecated in the next major version; use from trulens.apps.app import instrument
        instead.
  from trulens.apps.custom import instrument


✅ Libraries imported successfully


In [3]:
# Configuration
GENERATION_MODEL = "phi3:mini"
EMBEDDING_MODEL = "llama3.2"
OLLAMA_BASE_URL = "http://localhost:11434"

print(f"📋 Configuration:")
print(f"  Generation Model: {GENERATION_MODEL}")
print(f"  Embedding Model: {EMBEDDING_MODEL}")
print(f"  Ollama URL: {OLLAMA_BASE_URL}")

📋 Configuration:
  Generation Model: phi3:mini
  Embedding Model: llama3.2
  Ollama URL: http://localhost:11434


In [4]:
# Helper function for Ollama calls
def call_ollama(prompt: str, model: str = GENERATION_MODEL, temperature: float = 0.0, **kwargs) -> str:
    """Call Ollama for text generation"""
    try:
        response = completion(
            model=f"ollama/{model}",
            messages=[{"role": "user", "content": prompt}],
            api_base=OLLAMA_BASE_URL,
            temperature=temperature,
            stop=["###", "User:", "Question:", "\n\n\n"],
            **kwargs
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"

print("✅ Ollama helper function defined")

✅ Ollama helper function defined


In [5]:
# Setup TruLens provider for evaluations
trulens_provider = OpenAIProvider(
    model_engine=GENERATION_MODEL,
    base_url=f"{OLLAMA_BASE_URL}/v1",
    api_key="ollama"  # Dummy key for local Ollama
)

print("✅ TruLens provider configured")

✅ TruLens provider configured


---
# Section 2: Context Relevance {#section-2}

## What is Context Relevance?

**Context Relevance** measures whether the retrieved documents/chunks are actually relevant to answering the user's query. This is the **first line of defense** against RAG failures.

### Why It Matters

Imagine asking "What is our vacation policy?" and the retrieval system returns:
- ✅ Document about vacation days (RELEVANT)
- ❌ Document about office parking (IRRELEVANT)
- ❌ Document about salary structure (IRRELEVANT)

If irrelevant context is passed to the LLM, it may:
1. Get confused and generate incorrect answers
2. Mix irrelevant information into the response
3. Hallucinate connections between unrelated topics

### How It's Computed

**Step-by-step process:**

1. **Input**: Query + Retrieved Context Chunk
2. **LLM Evaluation**: "Is this context relevant to answering the query?"
3. **Output**: Score from 0.0 (not relevant) to 1.0 (highly relevant)
4. **Aggregation**: Average across all retrieved chunks

**Formula**:
```
Context_Relevance = Σ(relevance_score_i) / n_chunks
```

### Interpretation

- **0.8 - 1.0**: Excellent retrieval, highly relevant contexts
- **0.5 - 0.8**: Good, but some irrelevant chunks mixed in
- **0.0 - 0.5**: Poor retrieval, mostly irrelevant contexts

### Example

**Query**: "How many vacation days do employees get?"

**Retrieved Contexts**:
1. "Employees receive 20 days of paid vacation annually" → **Score: 1.0** ✅
2. "Health insurance covers medical and dental" → **Score: 0.0** ❌
3. "Vacation requests must be submitted 2 weeks in advance" → **Score: 0.9** ✅

**Average Context Relevance**: (1.0 + 0.0 + 0.9) / 3 = **0.63**

In [6]:
# Create sample company policy documents
company_policies = [
    "Employees are entitled to 20 days of paid vacation per year. Vacation requests must be submitted at least 2 weeks in advance through the HR portal.",
    
    "The company provides comprehensive health insurance covering medical, dental, and vision care. Coverage begins on the first day of employment.",
    
    "Remote work policy allows employees to work from home up to 3 days per week. Employees must coordinate with their team and manager. Core hours are 10 AM - 4 PM local time.",
    
    "Annual performance reviews are conducted in Q1 of each year. Reviews include goal setting, feedback discussion, and compensation adjustments.",
    
    "The company matches 401(k) contributions up to 6% of salary. Employees are immediately vested in their contributions.",
]

print("✅ Company policy documents created")
print(f"   Total documents: {len(company_policies)}")

✅ Company policy documents created
   Total documents: 5


In [7]:
# %%  FINAL GUARANTEED FIX — wipes all Chroma collections before rebuilding
import requests
import chromadb
from chromadb.config import Settings

OLLAMA_BASE_URL = "http://localhost:11434"
EMBEDDING_MODEL = "llama3.2"
COLLECTION_NAME = "company_policies_triad"

# 1️⃣ Create brand-new, non-persistent Chroma client
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False, is_persistent=False))
print("🧹 Started fresh in-memory Chroma client.")

# 2️⃣ Delete *all* existing collections (safeguard)
collections = chroma_client.list_collections()
if collections:
    print("🧼 Removing all existing collections:")
    for col in collections:
        try:
            chroma_client.delete_collection(col.name)
            print(f"   - Deleted '{col.name}'")
        except Exception as e:
            print(f"   ⚠️ Could not delete {col.name}: {e}")
else:
    print("ℹ️ No existing collections found.")

# 3️⃣ Helper to get embeddings
def get_embedding(text: str):
    url = f"{OLLAMA_BASE_URL}/api/embeddings"
    payload = {"model": EMBEDDING_MODEL, "prompt": text}
    response = requests.post(url, json=payload)
    response.raise_for_status()
    return response.json()["embedding"]

# 4️⃣ Check embedding dimension
test_emb = get_embedding("dimension check")
print(f"✅ Embedding dimension for '{EMBEDDING_MODEL}': {len(test_emb)}")

# 5️⃣ Create new collection
collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"description": f"Company HR policies for RAG Triad evaluation using {EMBEDDING_MODEL}"}
)
print(f"✅ Created new collection '{COLLECTION_NAME}'")

# 6️⃣ Add company policy docs
print("\n📄 Adding documents...")
for i, doc in enumerate(company_policies):
    emb = get_embedding(doc)
    collection.add(embeddings=[emb], documents=[doc], ids=[f"doc_{i}"])
    print(f"  ✓ Added document {i+1}/{len(company_policies)}")

print("\n✅ Collection ready for querying.")


🧹 Started fresh in-memory Chroma client.
ℹ️ No existing collections found.
✅ Embedding dimension for 'llama3.2': 3072
✅ Created new collection 'company_policies_triad'

📄 Adding documents...
  ✓ Added document 1/5
  ✓ Added document 2/5
  ✓ Added document 3/5
  ✓ Added document 4/5
  ✓ Added document 5/5

✅ Collection ready for querying.


In [8]:
# Create Context Relevance feedback function
def create_context_relevance_feedback():
    """
    Evaluates if retrieved context is relevant to the query.
    Returns a score between 0.0 (not relevant) and 1.0 (highly relevant).
    """
    def context_relevance(query: str, context: str) -> float:
        prompt = f"""Rate how relevant this context is to answering the query.

Query: {query}

Context: {context}

Is the context relevant for answering this query? Consider:
- Does it contain information that helps answer the query?
- Is the topic related to what's being asked?

Respond with ONLY a number between 0.0 and 1.0:
- 0.0 = Completely irrelevant
- 0.5 = Somewhat relevant
- 1.0 = Highly relevant and directly answers the query

Score:"""
        
        response = call_ollama(prompt, temperature=0.0)
        
        try:
            # Extract numeric score
            score_str = response.strip().split()[0].replace(',', '.')
            score = float(score_str)
            return max(0.0, min(1.0, score))
        except:
            print(f"  ⚠️ Could not parse score from: {response}")
            return 0.5  # Default to neutral
    
    return Feedback(context_relevance, name="Context Relevance")

f_context_relevance = create_context_relevance_feedback()
print("✅ Context Relevance feedback function created")

Feedback implementation <function create_context_relevance_feedback.<locals>.context_relevance at 0x000001FCF8A0FD80> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


✅ Context Relevance feedback function created


In [9]:
# Demo: Context Relevance Evaluation
print("\n" + "="*80)
print("CONTEXT RELEVANCE DEMONSTRATION")
print("="*80)

test_query = "How many vacation days do employees get?"
print(f"\n📝 Query: {test_query}\n")

# 🔧 Embed the query using the same model as the documents
query_emb = get_embedding(test_query)

# Retrieve contexts using query_embeddings (NOT query_texts)
results = collection.query(
    query_embeddings=[query_emb],
    n_results=3
)

contexts = results['documents'][0]
relevance_scores = []

print("📊 Evaluating Context Relevance:\n")

for i, context in enumerate(contexts, 1):
    score = f_context_relevance.imp(test_query, context)
    relevance_scores.append(score)
    
    print(f"Context {i}:")
    print(f"  Text: {context[:100]}...")
    print(f"  Relevance Score: {score:.2f}")
    print(f"  Assessment: {'✅ RELEVANT' if score >= 0.7 else '⚠️ SOMEWHAT RELEVANT' if score >= 0.4 else '❌ IRRELEVANT'}")
    print()

avg_relevance = np.mean(relevance_scores)
print(f"📈 Average Context Relevance: {avg_relevance:.2f}")
print(f"   Interpretation: {'✅ Excellent' if avg_relevance >= 0.8 else '✓ Good' if avg_relevance >= 0.6 else '⚠️ Needs Improvement'}")
print("="*80)



CONTEXT RELEVANCE DEMONSTRATION

📝 Query: How many vacation days do employees get?

📊 Evaluating Context Relevance:

Context 1:
  Text: Employees are entitled to 20 days of paid vacation per year. Vacation requests must be submitted at ...
  Relevance Score: 1.00
  Assessment: ✅ RELEVANT

Context 2:
  Text: The company provides comprehensive health insurance covering medical, dental, and vision care. Cover...
  Relevance Score: 0.20
  Assessment: ❌ IRRELEVANT

Context 3:
  Text: The company matches 401(k) contributions up to 6% of salary. Employees are immediately vested in the...
  Relevance Score: 0.50
  Assessment: ⚠️ SOMEWHAT RELEVANT

📈 Average Context Relevance: 0.57
   Interpretation: ⚠️ Needs Improvement


---
# Section 3: Groundedness (Faithfulness) {#section-3}

## What is Groundedness?

**Groundedness** (also called Faithfulness) measures whether the LLM's response is **supported by the retrieved context**. This metric detects hallucinations where the model makes up information not present in the source documents.

### Why It Matters

Even with relevant context, LLMs can:
- **Exaggerate**: "up to 20 days" becomes "unlimited vacation"
- **Invent details**: Add requirements not mentioned in policies
- **Conflate information**: Mix facts from different contexts incorrectly

### How It's Computed

**Step-by-step process:**

1. **Break Down Answer**: Split the response into individual claims/statements
2. **For Each Claim**: Check if it's supported by the retrieved context
3. **LLM Verification**: "Can this claim be verified from the context?"
4. **Scoring**: Calculate ratio of supported claims to total claims

**Formula**:
```
Groundedness = (Number of Supported Claims) / (Total Claims)
```

### Example Breakdown

**Context**: "Employees get 20 days of paid vacation per year."

**Response**: "Employees receive 20 days of vacation annually and can carry over 5 days to next year."

**Claims**:
1. "Employees receive 20 days vacation" → ✅ **Supported** (found in context)
2. "Can carry over 5 days" → ❌ **NOT Supported** (not mentioned in context)

**Groundedness Score**: 1/2 = **0.50** (Only 50% grounded)

### Interpretation

- **0.9 - 1.0**: Excellent, all claims supported by context
- **0.7 - 0.9**: Good, most claims grounded
- **0.5 - 0.7**: Moderate, some hallucinations present
- **0.0 - 0.5**: Poor, significant hallucination issues

In [10]:
# Create Groundedness feedback function
def create_groundedness_feedback():
    """
    Evaluates if the answer is grounded in/supported by the retrieved context.
    Checks for hallucinations and unsupported claims.
    """
    def groundedness(context: str, answer: str) -> float:
        prompt = f"""Evaluate if the answer is fully supported by the context.

Context:
{context}

Answer:
{answer}

Check if ALL statements in the answer can be verified from the context.
Look for:
- Made-up facts not in context
- Exaggerated claims
- Information from outside the context

Respond with ONLY a number between 0.0 and 1.0:
- 0.0 = Answer contains made-up information, not supported by context
- 0.5 = Answer is partially supported, some claims cannot be verified
- 1.0 = Answer is fully supported, all claims found in context

Score:"""
        
        response = call_ollama(prompt, temperature=0.0)
        
        try:
            score_str = response.strip().split()[0].replace(',', '.')
            score = float(score_str)
            return max(0.0, min(1.0, score))
        except:
            print(f"  ⚠️ Could not parse score from: {response}")
            return 0.5
    
    return Feedback(groundedness, name="Groundedness")

f_groundedness = create_groundedness_feedback()
print("✅ Groundedness feedback function created")

Feedback implementation <function create_groundedness_feedback.<locals>.groundedness at 0x000001FCAC63CB80> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


✅ Groundedness feedback function created


In [11]:
# %%
# Demo: Groundedness Evaluation
print("\n" + "="*80)
print("GROUNDEDNESS DEMONSTRATION")
print("="*80)

query = "What is the vacation policy?"

# Get embedding and retrieve - but let's search for vacation specifically
results = collection.query(
    query_embeddings=[get_embedding("vacation days policy request")],  # More specific query
    n_results=1
)
context = results['documents'][0][0]

# Verify we got the right context
print(f"📄 Retrieved Context:")
print(f"{context}\n")

# Check if it's about vacation
if "vacation" not in context.lower():
    print("⚠️ WARNING: Retrieved context doesn't mention vacation!")
    print("Searching for correct document...\n")
    
    # Manual fallback - find the vacation policy document
    for doc in company_policies:
        if "vacation" in doc.lower():
            context = doc
            print(f"✅ Found correct context:\n{context}\n")
            break

print("-"*80)

# Create different quality answers
test_cases = [
    {
        "name": "Well-Grounded Answer",
        "answer": "Employees are entitled to 20 days of paid vacation per year. Requests must be submitted at least 2 weeks in advance through the HR portal.",
        "expected": "High (0.9-1.0)"
    },
    {
        "name": "Partially Grounded",
        "answer": "Employees get 20 days vacation per year. Unused vacation days can be carried over to the next year.",
        "expected": "Medium (0.5-0.7)"
    },
    {
        "name": "Hallucinated Answer",
        "answer": "Employees get unlimited vacation days and can take time off whenever they want without approval.",
        "expected": "Low (0.0-0.3)"
    }
]

for i, test in enumerate(test_cases, 1):
    print(f"\n{i}. {test['name']}")
    print(f"   Answer: {test['answer']}")
    
    score = f_groundedness.imp(context, test['answer'])
    
    print(f"   Groundedness Score: {score:.2f}")
    print(f"   Expected Range: {test['expected']}")
    print(f"   Assessment: {'✅ GROUNDED' if score >= 0.8 else '⚠️ PARTIALLY GROUNDED' if score >= 0.5 else '❌ HALLUCINATED'}")

print("\n" + "="*80)


GROUNDEDNESS DEMONSTRATION
📄 Retrieved Context:
Annual performance reviews are conducted in Q1 of each year. Reviews include goal setting, feedback discussion, and compensation adjustments.

⚠️ WARNING: Retrieved context doesn't mention vacation!
Searching for correct document...

✅ Found correct context:
Employees are entitled to 20 days of paid vacation per year. Vacation requests must be submitted at least 2 weeks in advance through the HR portal.

--------------------------------------------------------------------------------

1. Well-Grounded Answer
   Answer: Employees are entitled to 20 days of paid vacation per year. Requests must be submitted at least 2 weeks in advance through the HR portal.
   Groundedness Score: 1.00
   Expected Range: High (0.9-1.0)
   Assessment: ✅ GROUNDED

2. Partially Grounded
   Answer: Employees get 20 days vacation per year. Unused vacation days can be carried over to the next year.
  ⚠️ Could not parse score from: The answer does NOT contain any 

---
# Section 4: Answer Relevance {#section-4}

## What is Answer Relevance?

**Answer Relevance** measures whether the generated response actually addresses the original question. Even with relevant context and grounded answers, the response might still fail to answer what was asked.

### Why It Matters

A response can be:
- **Factually correct** (grounded in context)
- **Using relevant sources** (high context relevance)
- But **still not answer the question**

### Common Failure Modes

1. **Partial Answer**: Only addresses part of the question
2. **Tangential Response**: Talks about related topics but misses the point
3. **Over-General**: Provides generic information instead of specific answer
4. **Missing Key Info**: Leaves out critical details needed to answer

### How It's Computed

**Step-by-step process:**

1. **Input**: Original Query + Generated Answer
2. **LLM Evaluation**: "Does this answer address what was asked?"
3. **Scoring**: Rate completeness and directness of response
4. **Output**: Score from 0.0 (irrelevant) to 1.0 (perfectly relevant)

### Example

**Question**: "How many vacation days do I get and when can I use them?"

**Answer Options**:

1. **High Relevance (1.0)**: ✅
   "You get 20 days of paid vacation per year. You can use them after submitting a request 2 weeks in advance through the HR portal."
   → Answers BOTH parts: how many + when

2. **Medium Relevance (0.5)**: ⚠️
   "Employees are entitled to 20 days of paid vacation annually."
   → Only answers "how many", missing "when can use"

3. **Low Relevance (0.2)**: ❌
   "Our company has competitive benefits including health insurance and vacation time."
   → Vague, doesn't answer the specific question

### Interpretation

- **0.9 - 1.0**: Excellent, directly and completely answers the question
- **0.7 - 0.9**: Good, addresses main points but may miss details
- **0.5 - 0.7**: Moderate, partially relevant but incomplete
- **0.0 - 0.5**: Poor, doesn't address the question

In [12]:
# %%
import re

def parse_score(response: str) -> float:
    """Robust score parser used by all feedback functions"""
    try:
        # Strategy 1: Find first number between 0-1
        match = re.search(r'\b([0-1]\.?\d*)\b', response)
        if match:
            score = float(match.group(1))
            if 0.0 <= score <= 1.0:
                return score
        
        # Strategy 2: Look after "Score:"
        if "score:" in response.lower():
            after_score = response.lower().split("score:")[-1]
            match = re.search(r'[0-1]\.?\d*', after_score)
            if match:
                return float(match.group())
        
    except:
        pass
    
    # Fallback: Keyword analysis
    resp_lower = response.lower()
    if any(word in resp_lower for word in ["1.0", "completely", "fully", "all"]):
        return 1.0
    elif any(word in resp_lower for word in ["0.0", "not at all", "does not"]):
        return 0.0
    else:
        return 0.5

# Context Relevance
def create_context_relevance_feedback():
    def context_relevance(query: str, context: str) -> float:
        prompt = f"""Query: {query}
Context: {context}

Rate relevance 0.0-1.0. Number only:"""
        response = call_ollama(prompt, temperature=0.0, max_tokens=10)
        return parse_score(response)
    return Feedback(context_relevance, name="Context Relevance")

# Groundedness
def create_groundedness_feedback():
    def groundedness(context: str, answer: str) -> float:
        prompt = f"""Context: {context}
Answer: {answer}

Rate 0.0-1.0 if answer is supported by context. Number only:"""
        response = call_ollama(prompt, temperature=0.0, max_tokens=10)
        return parse_score(response)
    return Feedback(groundedness, name="Groundedness")

# Answer Relevance
def create_answer_relevance_feedback():
    def answer_relevance(query: str, answer: str) -> float:
        prompt = f"""Question: {query}
Answer: {answer}

Rate 0.0-1.0 if answer addresses question. Number only:"""
        response = call_ollama(prompt, temperature=0.0, max_tokens=10)
        return parse_score(response)
    return Feedback(answer_relevance, name="Answer Relevance")

# Create all feedback functions
f_context_relevance = create_context_relevance_feedback()
f_groundedness = create_groundedness_feedback()
f_answer_relevance = create_answer_relevance_feedback()

print("✅ All feedback functions created with robust parsing")

Feedback implementation <function create_context_relevance_feedback.<locals>.context_relevance at 0x000001FCF8990EA0> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.
Feedback implementation <function create_groundedness_feedback.<locals>.groundedness at 0x000001FCF8A0FD80> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.
Feedback implementation <function create_answer_relevance_feedback.<locals>.answer_relevance at 0x000001FCAC63CB80> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.


✅ All feedback functions created with robust parsing


In [13]:
# Demo: Answer Relevance Evaluation
print("\n" + "="*80)
print("ANSWER RELEVANCE DEMONSTRATION")
print("="*80)

query = "How many vacation days do employees get and what is the process to request time off?"

test_answers = [
    {
        "name": "Complete Answer",
        "answer": "Employees are entitled to 20 days of paid vacation per year. To request time off, submit your request at least 2 weeks in advance through the HR portal.",
        "expected": "High (0.9-1.0)"
    },
    {
        "name": "Partial Answer",
        "answer": "Employees receive 20 days of paid vacation annually.",
        "expected": "Medium (0.5-0.7)"
    },
    {
        "name": "Off-Topic Answer",
        "answer": "Our company offers competitive benefits including health insurance, 401k matching, and various leave policies.",
        "expected": "Low (0.0-0.3)"
    }
]

print(f"\n❓ Question: {query}\n")
print("-"*80)

for i, test in enumerate(test_answers, 1):
    print(f"\n{i}. {test['name']}")
    print(f"   Answer: {test['answer']}")
    
    score = f_answer_relevance.imp(query, test['answer'])
    
    print(f"   Answer Relevance Score: {score:.2f}")
    print(f"   Expected Range: {test['expected']}")
    print(f"   Assessment: {'✅ RELEVANT' if score >= 0.8 else '⚠️ PARTIALLY RELEVANT' if score >= 0.5 else '❌ IRRELEVANT'}")

print("\n" + "="*80)


ANSWER RELEVANCE DEMONSTRATION

❓ Question: How many vacation days do employees get and what is the process to request time off?

--------------------------------------------------------------------------------

1. Complete Answer
   Answer: Employees are entitled to 20 days of paid vacation per year. To request time off, submit your request at least 2 weeks in advance through the HR portal.
   Answer Relevance Score: 1.00
   Expected Range: High (0.9-1.0)
   Assessment: ✅ RELEVANT

2. Partial Answer
   Answer: Employees receive 20 days of paid vacation annually.
   Answer Relevance Score: 1.00
   Expected Range: Medium (0.5-0.7)
   Assessment: ✅ RELEVANT

3. Off-Topic Answer
   Answer: Our company offers competitive benefits including health insurance, 401k matching, and various leave policies.
   Answer Relevance Score: 1.00
   Expected Range: Low (0.0-0.3)
   Assessment: ✅ RELEVANT



---
# Section 5: Complete RAG Evaluation {#section-5}

## Building a Complete RAG System with All Triad Metrics

Now we'll build a full RAG application and evaluate it using **all three RAG Triad metrics together**. This gives us a comprehensive view of system quality.

### What We'll Build

1. **RAG Application**: Query → Retrieve → Generate
2. **Evaluation Pipeline**: Track all 3 metrics for each query
3. **Results Dashboard**: Aggregate and per-query metrics

### Interpreting Combined Results

**Scenario Analysis**:

| Context Rel | Groundedness | Answer Rel | Diagnosis |
|-------------|--------------|------------|-----------|
| ✅ High | ✅ High | ✅ High | **Perfect** - System working well |
| ❌ Low | ✅ High | ❌ Low | **Retrieval Issue** - Wrong docs retrieved |
| ✅ High | ❌ Low | ✅ High | **Hallucination** - Model inventing facts |
| ✅ High | ✅ High | ❌ Low | **Generation Issue** - Not answering question |
| ❌ Low | ❌ Low | ❌ Low | **System Failure** - Multiple problems |

### Score Interpretation

**Overall System Health**:
- **All ≥ 0.8**: Production ready ✅
- **All ≥ 0.6**: Acceptable, monitor closely ⚠️
- **Any < 0.5**: Needs improvement ❌

In [14]:
# %%
class EvaluatedRAG:
    """RAG with better retrieval (top-1 only)"""
    
    def __init__(self, collection):
        self.collection = collection
        self.f_context_relevance = create_context_relevance_feedback()
        self.f_groundedness = create_groundedness_feedback()
        self.f_answer_relevance = create_answer_relevance_feedback()
        self.evaluation_results = []
    
    @instrument
    def retrieve(self, query: str, n_results: int = 1) -> List[str]:  # Changed to 1
        """Retrieve relevant documents"""
        query_emb = get_embedding(query)
        results = self.collection.query(
            query_embeddings=[query_emb],
            n_results=n_results  # Will default to 1
        )
        return results['documents'][0] if results['documents'] else []
    
    # ... rest of the class stays the same ...
    
    @instrument
    def generate_response(self, query: str, contexts: List[str]) -> str:
        """Generate answer from contexts"""
        if not contexts:
            return "I don't have enough information to answer this question."
        
        context_str = "\n\n".join([f"Context {i+1}: {ctx}" for i, ctx in enumerate(contexts)])
        
        prompt = f"""You are a helpful HR assistant. Answer the question using ONLY the information from the company policy documents below.

Company Policy Documents:
{context_str}

Question: {query}

Provide a clear, direct answer based only on the documents. Do not add information not present in the documents.

Answer:"""
        
        response = call_ollama(prompt, temperature=0.0)
        
        # Clean up any hallucinated continuations
        stop_markers = ["### User:", "### Assistant:", "\n\nUser:", "\n\nQuestion:", "\n\nNow,"]
        cleaned_response = response
        for marker in stop_markers:
            if marker in cleaned_response:
                cleaned_response = cleaned_response.split(marker)[0].strip()
        
        return cleaned_response
    
    def evaluate_query(self, query: str) -> Dict[str, Any]:
        """
        Execute full RAG pipeline with evaluation.
        Returns metrics for all three aspects of RAG Triad.
        """
        # 1. Retrieve contexts
        contexts = self.retrieve(query)
        
        if not contexts:
            return {
                "query": query,
                "answer": "No relevant context found.",
                "contexts": [],
                "context_relevance": 0.0,
                "groundedness": 0.0,
                "answer_relevance": 0.0,
                "overall_score": 0.0
            }
        
        # 2. Evaluate Context Relevance (for each retrieved context)
        context_scores = []
        for ctx in contexts:
            score = self.f_context_relevance.imp(query, ctx)
            context_scores.append(score)
        
        avg_context_relevance = np.mean(context_scores)
        
        # 3. Generate answer
        answer = self.generate_response(query, contexts)
        
        # 4. Evaluate Groundedness
        combined_context = "\n\n".join(contexts)
        groundedness_score = self.f_groundedness.imp(combined_context, answer)
        
        # 5. Evaluate Answer Relevance
        answer_relevance_score = self.f_answer_relevance.imp(query, answer)
        
        # 6. Calculate overall score (average of 3 metrics)
        overall_score = np.mean([avg_context_relevance, groundedness_score, answer_relevance_score])
        
        result = {
            "query": query,
            "answer": answer,
            "contexts": contexts,
            "context_relevance": avg_context_relevance,
            "groundedness": groundedness_score,
            "answer_relevance": answer_relevance_score,
            "overall_score": overall_score,
            "individual_context_scores": context_scores
        }
        
        self.evaluation_results.append(result)
        return result
    
    def get_aggregate_metrics(self) -> Dict[str, float]:
        """Calculate aggregate metrics across all evaluations"""
        if not self.evaluation_results:
            return {}
        
        return {
            "avg_context_relevance": np.mean([r["context_relevance"] for r in self.evaluation_results]),
            "avg_groundedness": np.mean([r["groundedness"] for r in self.evaluation_results]),
            "avg_answer_relevance": np.mean([r["answer_relevance"] for r in self.evaluation_results]),
            "avg_overall_score": np.mean([r["overall_score"] for r in self.evaluation_results]),
            "total_queries": len(self.evaluation_results)
        }

print("✅ EvaluatedRAG class defined")

✅ EvaluatedRAG class defined


In [15]:
# Demo: Complete RAG Evaluation with All Metrics
print("\n" + "="*80)
print("COMPLETE RAG TRIAD EVALUATION")
print("="*80)

# Initialize RAG system
rag_system = EvaluatedRAG(collection)

# Test queries covering different scenarios
test_queries = [
    "How many vacation days do employees get?",
    "What is the remote work policy?",
    "Tell me about health insurance benefits",
    "What is the 401k matching policy?",
]

print("\n📊 Evaluating RAG System on Test Queries\n")
print("="*80)

# Evaluate each query
for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*80}")
    print(f"Query {i}/{len(test_queries)}: {query}")
    print('='*80)
    
    result = rag_system.evaluate_query(query)
    
    print(f"\n✅ Answer:\n{result['answer']}\n")
    
    print("📈 RAG Triad Scores:")
    print(f"  • Context Relevance:  {result['context_relevance']:.2f} {'✅' if result['context_relevance'] >= 0.7 else '⚠️' if result['context_relevance'] >= 0.5 else '❌'}")
    print(f"  • Groundedness:       {result['groundedness']:.2f} {'✅' if result['groundedness'] >= 0.7 else '⚠️' if result['groundedness'] >= 0.5 else '❌'}")
    print(f"  • Answer Relevance:   {result['answer_relevance']:.2f} {'✅' if result['answer_relevance'] >= 0.7 else '⚠️' if result['answer_relevance'] >= 0.5 else '❌'}")
    print(f"  • Overall Score:      {result['overall_score']:.2f} {'✅' if result['overall_score'] >= 0.7 else '⚠️' if result['overall_score'] >= 0.5 else '❌'}")
    
    print(f"\n📄 Retrieved Contexts: {len(result['contexts'])}")
    for j, ctx in enumerate(result['contexts'][:2], 1):  # Show first 2
        print(f"  {j}. {ctx[:80]}... (Score: {result['individual_context_scores'][j-1]:.2f})")

Feedback implementation <function create_context_relevance_feedback.<locals>.context_relevance at 0x000001FCF8990860> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.
Feedback implementation <function create_groundedness_feedback.<locals>.groundedness at 0x000001FCAC63FC40> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.
Feedback implementation <function create_answer_relevance_feedback.<locals>.answer_relevance at 0x000001FCF89914E0> cannot be serialized: Module __main__ is not importable. This may be ok unless you are using the deferred feedback mode.



COMPLETE RAG TRIAD EVALUATION

📊 Evaluating RAG System on Test Queries


Query 1/4: How many vacation days do employees get?

✅ Answer:
Employees are entitled to 20 days of paid vacation per year.

📈 RAG Triad Scores:
  • Context Relevance:  1.00 ✅
  • Groundedness:       1.00 ✅
  • Answer Relevance:   1.00 ✅
  • Overall Score:      1.00 ✅

📄 Retrieved Contexts: 1
  1. Employees are entitled to 20 days of paid vacation per year. Vacation requests m... (Score: 1.00)

Query 2/4: What is the remote work policy?

✅ Answer:
The provided company policy documents do not contain specific details regarding the remote work policy. Therefore, I cannot provide an answer about it from these documents alone.

📈 RAG Triad Scores:
  • Context Relevance:  0.20 ❌
  • Groundedness:       0.50 ⚠️
  • Answer Relevance:   0.00 ❌
  • Overall Score:      0.23 ❌

📄 Retrieved Contexts: 1
  1. Annual performance reviews are conducted in Q1 of each year. Reviews include goa... (Score: 0.20)

Query 3/4: Tell me a

---
# Section 6: Comparison with RAGAS {#section-6}

## TruLens vs RAGAS: Side-by-Side Comparison

Both TruLens and RAGAS are popular frameworks for evaluating RAG systems. Here's how they compare:

### Metric Comparison

| Metric | TruLens | RAGAS | Description |
|--------|---------|-------|-------------|
| **Context Relevance** | ✅ Yes | ✅ Context Precision | Relevance of retrieved docs |
| **Groundedness** | ✅ Yes | ✅ Faithfulness | Answer supported by context |
| **Answer Relevance** | ✅ Yes | ✅ Answer Relevancy | Answer addresses question |
| **Context Recall** | ❌ No | ✅ Yes | All relevant docs retrieved |
| **Context Precision** | ❌ No | ✅ Yes | Precision of retrieval |

### Philosophical Differences

**TruLens (RAG Triad)**:
- Focus: **Minimal set of metrics** covering main failure modes
- Approach: **LLM-as-a-judge** for all evaluations
- Philosophy: 3 metrics are sufficient to catch all major issues
- Strength: Simplicity, interpretability

**RAGAS**:
- Focus: **Comprehensive metric suite** with granular measurements
- Approach: **Mix of LLM + heuristic** methods
- Philosophy: More metrics = more insights
- Strength: Detailed analysis, multiple perspectives

### Detailed Feature Comparison

| Feature | TruLens | RAGAS |
|---------|---------|-------|
| **Evaluation Method** | LLM-based | LLM + Heuristics |
| **Built-in Dashboard** | ✅ Streamlit UI | ❌ External needed |
| **Reasoning Transparency** | ✅ Chain-of-thought | ⚠️ Limited |
| **Framework Support** | LangChain, LlamaIndex, Custom | LangChain, LlamaIndex |
| **Ground Truth Required** | ❌ No | ⚠️ Optional but recommended |
| **Setup Complexity** | Simple | Moderate |
| **Customization** | High | Moderate |
| **Production Monitoring** | ✅ Strong | ⚠️ Limited |

### When to Use Which?

**Use TruLens When:**
- ✅ You want **simple, interpretable metrics**
- ✅ You need **production monitoring** with dashboards
- ✅ You prefer **pure LLM-based evaluation**
- ✅ You want **quick setup** and iteration
- ✅ You need **detailed reasoning** for each score

**Use RAGAS When:**
- ✅ You want **comprehensive metric coverage**
- ✅ You have **ground truth data** available
- ✅ You prefer **hybrid evaluation** (LLM + heuristics)
- ✅ You need **research-grade evaluation**
- ✅ You want **granular retrieval metrics**

### Can They Be Used Together?

**Yes!** Many teams use both:

1. **Development**: Use RAGAS for comprehensive analysis
   - Deep dive into system behavior
   - Identify specific failure modes
   - Compare multiple configurations

2. **Production**: Use TruLens for monitoring
   - Track 3 key metrics over time
   - Dashboard for stakeholders
   - Quick identification of issues

### Example Combined Workflow

```python
# Development phase
ragas_scores = ragas.evaluate(
    dataset=test_dataset,
    metrics=[context_precision, context_recall, faithfulness, answer_relevancy]
)
# Detailed analysis with 4+ metrics

# Production phase
with tru_recorder as recording:
    response = rag_app.query(user_question)
# Monitor Context Relevance, Groundedness, Answer Relevance
```

### Metric Mapping

| TruLens | RAGAS Equivalent |
|---------|------------------|
| Context Relevance | Context Precision |
| Groundedness | Faithfulness |
| Answer Relevance | Answer Relevancy |
| - | Context Recall |

### Summary

- **TruLens**: Focused, production-oriented, great UX
- **RAGAS**: Comprehensive, research-oriented, more metrics
- **Best Practice**: Use both in different stages of development

In [16]:
# Comparison Demo: Show TruLens metric interpretation
print("\n" + "="*80)
print("TRULENS vs RAGAS METRICS - INTERPRETATION GUIDE")
print("="*80)

comparison_data = {
    "TruLens": {
        "Context Relevance": 0.85,
        "Groundedness": 0.92,
        "Answer Relevance": 0.88,
    },
    "RAGAS (Equivalent)": {
        "Context Precision": 0.84,  # Similar to Context Relevance
        "Faithfulness": 0.91,       # Similar to Groundedness
        "Answer Relevancy": 0.87,   # Similar to Answer Relevance
        "Context Recall": 0.79      # TruLens doesn't measure this
    }
}

print("\n📊 Example Evaluation Results:\n")

print("TruLens RAG Triad:")
for metric, score in comparison_data["TruLens"].items():
    print(f"  • {metric:20s}: {score:.2f} {'✅' if score >= 0.8 else '⚠️'}")

print("\nRAGAS Metrics:")
for metric, score in comparison_data["RAGAS (Equivalent)"].items():
    print(f"  • {metric:20s}: {score:.2f} {'✅' if score >= 0.8 else '⚠️'}")

print("\n💡 Key Observations:")
print("  • Scores are generally similar between frameworks")
print("  • RAGAS provides additional Context Recall metric")
print("  • TruLens focuses on 3 core metrics for simplicity")
print("  • Both frameworks effective at catching issues")

print("\n" + "="*80)


TRULENS vs RAGAS METRICS - INTERPRETATION GUIDE

📊 Example Evaluation Results:

TruLens RAG Triad:
  • Context Relevance   : 0.85 ✅
  • Groundedness        : 0.92 ✅
  • Answer Relevance    : 0.88 ✅

RAGAS Metrics:
  • Context Precision   : 0.84 ✅
  • Faithfulness        : 0.91 ✅
  • Answer Relevancy    : 0.87 ✅
  • Context Recall      : 0.79 ⚠️

💡 Key Observations:
  • Scores are generally similar between frameworks
  • RAGAS provides additional Context Recall metric
  • TruLens focuses on 3 core metrics for simplicity
  • Both frameworks effective at catching issues



---
# Section 7: Best Practices {#section-7}

## How to Improve Low RAG Triad Scores

### Problem: Low Context Relevance (< 0.6)

**Root Causes:**
- Poor retrieval algorithm
- Inadequate embeddings
- Wrong chunk size
- Insufficient reranking

**Solutions:**

1. **Improve Retrieval**:
   ```python
   # Increase k and rerank
   results = collection.query(query, n_results=10)  # Get more candidates
   reranked = rerank_by_relevance(results, query)   # Rerank top results
   contexts = reranked[:3]  # Keep top 3
   ```

2. **Better Embeddings**:
   - Use domain-specific embedding models
   - Fine-tune embeddings on your data
   - Try different embedding models

3. **Optimize Chunk Size**:
   ```python
   # Experiment with chunk sizes
   chunk_sizes = [256, 512, 1024]
   for size in chunk_sizes:
       evaluate_with_chunk_size(size)
   ```

4. **Add Metadata Filtering**:
   ```python
   # Filter by document type, date, etc.
   results = collection.query(
       query,
       where={"doc_type": "policy", "date": {"$gte": "2024-01-01"}}
   )
   ```

### Problem: Low Groundedness (< 0.7)

**Root Causes:**
- Model hallucination tendency
- Insufficient context
- Poor prompt engineering
- Model too creative (high temperature)

**Solutions:**

1. **Stronger Prompts**:
   ```python
   prompt = f"""Answer using ONLY the context below. Do NOT add any information 
   not explicitly stated in the context. If the context doesn't contain the answer,
   say "I don't have enough information."
   
   Context: {context}
   Question: {query}
   Answer:"""
   ```

2. **Lower Temperature**:
   ```python
   response = call_ollama(prompt, temperature=0.0)  # More deterministic
   ```

3. **Citation Requirements**:
   ```python
   prompt = f"""Answer the question and cite which context supports each claim.
   
   Context 1: {ctx1}
   Context 2: {ctx2}
   
   Question: {query}
   Format: [Answer] (Source: Context X)"""
   ```

4. **Post-Generation Verification**:
   ```python
   def verify_answer(answer, context):
       claims = extract_claims(answer)
       for claim in claims:
           if not is_supported(claim, context):
               return "I cannot verify all claims. Please check sources."
       return answer
   ```

### Problem: Low Answer Relevance (< 0.6)

**Root Causes:**
- Model misunderstanding query intent
- Answer too general/vague
- Missing key information
- Off-topic generation

**Solutions:**

1. **Query Preprocessing**:
   ```python
   def enhance_query(query):
       # Add context or rephrase
       enhanced = f"Based on company policies: {query}"
       return enhanced
   ```

2. **Answer Format Specification**:
   ```python
   prompt = f"""Answer the question directly and completely.
   
   Question: {query}
   Context: {context}
   
   Provide:
   1. Direct answer to the question
   2. Relevant details from context
   3. Keep it concise and on-topic
   
   Answer:"""
   ```

3. **Post-Processing**:
   ```python
   def ensure_relevance(query, answer):
       # Check if answer addresses query
       if not addresses_query(query, answer):
           return regenerate_with_stricter_prompt(query)
       return answer
   ```

## Production Monitoring Best Practices

### 1. Set Alert Thresholds

```python
# Define acceptable thresholds
THRESHOLDS = {
    "context_relevance": 0.7,
    "groundedness": 0.8,
    "answer_relevance": 0.7,
}

def check_quality(scores):
    alerts = []
    for metric, threshold in THRESHOLDS.items():
        if scores[metric] < threshold:
            alerts.append(f"⚠️ {metric} below threshold: {scores[metric]:.2f}")
    return alerts
```

### 2. Track Metrics Over Time

```python
# Log metrics for trend analysis
evaluation_log = {
    "timestamp": datetime.now(),
    "query": query,
    "metrics": {
        "context_relevance": 0.85,
        "groundedness": 0.92,
        "answer_relevance": 0.88,
    }
}
# Store in database for analysis
```

### 3. Implement Sampling

```python
# Don't evaluate every query in production (cost)
EVAL_SAMPLE_RATE = 0.1  # Evaluate 10% of queries

if random.random() < EVAL_SAMPLE_RATE:
    metrics = evaluate_rag_triad(query, contexts, answer)
    log_metrics(metrics)
```

### 4. Human Review Queue

```python
# Flag low scores for human review
if any(score < 0.6 for score in metrics.values()):
    add_to_review_queue(query, answer, metrics)
```

## Common Pitfalls to Avoid

### ❌ Pitfall 1: Ignoring Low Scores
**Problem**: Seeing low scores but not investigating  
**Solution**: Set up alerts and review processes

### ❌ Pitfall 2: Over-Optimizing One Metric
**Problem**: Improving groundedness but sacrificing answer relevance  
**Solution**: Balance all three metrics, aim for consistent scores

### ❌ Pitfall 3: Not Testing Edge Cases
**Problem**: System works on common queries but fails on unusual ones  
**Solution**: Build diverse test set with edge cases

### ❌ Pitfall 4: Static Evaluation
**Problem**: Evaluating once during development  
**Solution**: Continuous evaluation in production

### ❌ Pitfall 5: Trusting Scores Blindly
**Problem**: Assuming high scores = perfect system  
**Solution**: Combine automated metrics with human review

## Recommended Workflow

```
1. Development
   ├─ Build RAG system
   ├─ Evaluate on test set
   └─ Iterate until scores ≥ 0.7

2. Testing
   ├─ Expand test set
   ├─ Test edge cases
   └─ Human review sample

3. Production
   ├─ Monitor metrics (sample)
   ├─ Set up alerts
   ├─ Review flagged queries
   └─ Regular audits

4. Continuous Improvement
   ├─ Analyze trends
   ├─ Address failure patterns
   └─ Update prompts/retrieval
```

In [18]:
# %%
# Best Practices Demo: Score Interpretation and Action Items
print("\n" + "="*80)
print("BEST PRACTICES: INTERPRETING SCORES & ACTION ITEMS")
print("="*80)

def diagnose_rag_system(context_rel: float, groundedness: float, answer_rel: float):
    """Diagnose RAG system issues based on RAG Triad scores"""
    
    print(f"\n📊 Current Scores:")
    print(f"  • Context Relevance:  {context_rel:.2f}")
    print(f"  • Groundedness:       {groundedness:.2f}")
    print(f"  • Answer Relevance:   {answer_rel:.2f}")
    
    print(f"\n🔍 Diagnosis:")
    
    # Scenario 1: All good
    if all(score >= 0.8 for score in [context_rel, groundedness, answer_rel]):
        print("  ✅ SYSTEM HEALTHY - All metrics in excellent range")
        print("\n💡 Recommendations:")
        print("  • Continue monitoring in production")
        print("  • Maintain current configuration")
        print("  • Document what's working well")
    
    # Scenario 2: Low context relevance
    elif context_rel < 0.6:
        print("  ❌ RETRIEVAL PROBLEM - Poor context relevance")
        print("\n💡 Action Items:")
        print("  1. Review retrieval algorithm (try hybrid search)")
        print("  2. Improve embeddings (fine-tune or use better model)")
        print("  3. Adjust chunk size and overlap")
        print("  4. Add reranking step")
        print("  5. Check if documents cover query topics")
    
    # Scenario 3: Low groundedness
    elif groundedness < 0.7:
        print("  ❌ HALLUCINATION PROBLEM - Low groundedness")
        print("\n💡 Action Items:")
        print("  1. Strengthen prompts (emphasize 'only from context')")
        print("  2. Lower temperature (reduce creativity)")
        print("  3. Add citation requirements")
        print("  4. Consider smaller, more focused model")
        print("  5. Implement post-generation verification")
    
    # Scenario 4: Low answer relevance
    elif answer_rel < 0.6:
        print("  ❌ GENERATION PROBLEM - Answer not relevant")
        print("\n💡 Action Items:")
        print("  1. Improve prompt clarity and instructions")
        print("  2. Preprocess queries for better understanding")
        print("  3. Specify answer format in prompt")
        print("  4. Add examples in prompt (few-shot)")
        print("  5. Post-process to ensure completeness")
    
    # Scenario 5: Multiple issues
    elif sum(1 for s in [context_rel, groundedness, answer_rel] if s < 0.6) >= 2:
        print("  ❌ MULTIPLE ISSUES - System needs major improvements")
        print("\n💡 Action Items:")
        print("  1. Review entire pipeline end-to-end")
        print("  2. Start with improving retrieval first")
        print("  3. Test with simpler queries initially")
        print("  4. Consider different model or approach")
        print("  5. Get human feedback on failures")
    
    # Scenario 6: Moderate scores
    else:
        print("  ⚠️ MODERATE PERFORMANCE - Room for improvement")
        print("\n💡 Action Items:")
        print("  1. Focus on lowest scoring metric first")
        print("  2. A/B test improvements")
        print("  3. Collect user feedback")
        print("  4. Iterate incrementally")
    
    print("\n" + "-"*80)

# Test different scenarios
scenarios = [
    ("Healthy System", 0.85, 0.92, 0.88),
    ("Retrieval Issues", 0.45, 0.85, 0.80),
    ("Hallucination Issues", 0.85, 0.55, 0.82),
    ("Answer Quality Issues", 0.80, 0.85, 0.50),
    ("Multiple Problems", 0.45, 0.52, 0.48),
]

for name, ctx, ground, ans in scenarios:
    print(f"\n{'='*80}")
    print(f"Scenario: {name}")
    print('='*80)
    diagnose_rag_system(ctx, ground, ans)

print("\n" + "="*80)



BEST PRACTICES: INTERPRETING SCORES & ACTION ITEMS

Scenario: Healthy System

📊 Current Scores:
  • Context Relevance:  0.85
  • Groundedness:       0.92
  • Answer Relevance:   0.88

🔍 Diagnosis:
  ✅ SYSTEM HEALTHY - All metrics in excellent range

💡 Recommendations:
  • Continue monitoring in production
  • Maintain current configuration
  • Document what's working well

--------------------------------------------------------------------------------

Scenario: Retrieval Issues

📊 Current Scores:
  • Context Relevance:  0.45
  • Groundedness:       0.85
  • Answer Relevance:   0.80

🔍 Diagnosis:
  ❌ RETRIEVAL PROBLEM - Poor context relevance

💡 Action Items:
  1. Review retrieval algorithm (try hybrid search)
  2. Improve embeddings (fine-tune or use better model)
  3. Adjust chunk size and overlap
  4. Add reranking step
  5. Check if documents cover query topics

--------------------------------------------------------------------------------

Scenario: Hallucination Issues

📊 Cur